In [1]:
import urllib.request
import json
import pymongo
import pandas as pd
import datetime

In [2]:
def save_earthquakes(earthquake_url, host_name, host_port):
    try:
        response = urllib.request.urlopen(earthquake_url)
    except urllib.error.URLError as e:
        if hasattr(e, 'reason'):
            print('We failed to reach a server.')
            print('Reason: ', e.reason)
        elif hasattr(e, 'code'):
            print('The server couldn\'t fulfill the request.')
            print('Error code: ', e.code)
    else:
        # the url request was successful - convert the response to a string
        json_string = response.read().decode('utf-8')

        # the json package loads() converts the string to python dictionaries and lists
        eq_json = json.loads(json_string)

        # from the json dictionary we get the title to print
        title = eq_json['metadata']['title']
        print('Collected data from', title)
        #  and we get the list of earthquakes
        quakelist = eq_json['features']

        # Connection to Mongo DB
        try:
            client=pymongo.MongoClient(host_name, host_port)
            print ("Connected successfully!!!")
        except pymongo.errors.ConnectionFailure as e:
           print ("Could not connect to MongoDB: %s" % e )
        else:

            # use database named usgs or create it if not there already
            eqdb = client.usgs
            # create collection named earthquakes or create it if not there already
            quakecoll = eqdb.earthquakes
            # add all the earthquakes to the list
            quakecoll.insert_many(quakelist)
            print("Added", len(quakelist), "to earthquakes collection in usgs database")
            # close the database connection
        client.close()

In [5]:
def get_earthquakes(host_name, host_port):
    quake_docs = []
    # Connection to Mongo DB
    try:
        client=pymongo.MongoClient(host_name, host_port)
        print ("Connected successfully!!!")
    except pymongo.errors.ConnectionFailure as e:
        print ("Could not connect to MongoDB: %s" % e )
    else:
        # use database named usgs or create it if not there already
        eqdb = client.usgs
        # create collection named earthquakes or create it if not there already
        quakecoll = eqdb.earthquakes
        # add all the earthquakes to the list
        for doc in quakecoll.find():
            place = doc["properties"]["place"]
            # grab unix timestamp in milliseconds
            unix_time_mil = doc["properties"]["time"]
            # convert to unix in seconds
            unix_time = unix_time_mil / 1000
            quake_docs.append([place,unix_time])
            
        # close the database connection
    client.close()
    quake_docs = pd.DataFrame(quake_docs)
    quake_docs.columns = ['Location','Time']
    quake_docs['Time'] = quake_docs.Time.apply(lambda unix_time: datetime.datetime.fromtimestamp(unix_time).isoformat())
    return quake_docs

In [6]:
def list_earthquakes(host_name, host_port):
    quake_docs = get_earthquakes(host_name, host_port)
    print(quake_docs)

In [7]:
# get the bbc rss feed of news stories and connect to it
earthquake_url = "http://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/significant_month.geojson"

# connect to a mongo client using a host name and port
# use localhost to connect to your own device and
# port 27017 by default

host_name = 'localhost'
host_port = 27017
save_earthquakes(earthquake_url, host_name, host_port)
list_earthquakes(host_name, host_port)

Collected data from USGS Significant Earthquakes, Past Month
Connected successfully!!!
Added 8 to earthquakes collection in usgs database
Connected successfully!!!
                              Location                        Time
0   11 km NE of Snoqualmie, Washington  2023-08-08T06:17:23.550000
1             23 km S of Dezhou, China  2023-08-05T14:33:58.159000
2      96 km ENE of Port-Olry, Vanuatu  2023-07-26T08:44:35.661000
3      2 km E of Chino Valley, Arizona  2023-07-23T16:52:06.293000
4     43 km S of Intipucá, El Salvador  2023-07-18T20:22:07.747000
..                                 ...                         ...
57            42 km W of Sola, Vanuatu  2023-08-16T08:47:40.512000
58                          Washington  2023-08-08T06:17:23.840000
59            23 km S of Dezhou, China  2023-08-05T14:33:58.159000
60     96 km ENE of Port-Olry, Vanuatu  2023-07-26T08:44:35.647000
61     2 km E of Chino Valley, Arizona  2023-07-23T16:52:06.293000

[62 rows x 2 columns]
